# 37-LS. 모음조화/충돌회피 LS 코퍼스 빈도 조회

37번 사전검색 결과(용언 어간 리스트)의 실제 사용 빈도를 LS 코퍼스에서 조회

## 입력
- 37번 결과 CSV: `vowel_harmony_collision_all_*.csv`
- LS 단어 빈도: `00_raw_data/02_nikl_ls/07_ALL_word_freq.csv` (102MB)
- LS 형태소 빈도: `00_raw_data/02_nikl_ls/08_ALL_morpheme_freq.csv`

## 출력
- 빈도 보강 CSV: `37_vowel_harmony_collision/search_results/ls_freq_vowel_harmony_collision_*.csv`

## 1. 환경 설정

In [1]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# 경로 설정
SEARCH_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
LS_WORD_FREQ = f'{PROJECT_ROOT}/00_raw_data/02_nikl_ls/07_ALL_word_freq.csv'
LS_MORPH_FREQ = f'{PROJECT_ROOT}/00_raw_data/02_nikl_ls/08_ALL_morpheme_freq.csv'
RESULT_DIR = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

print(f'37번 결과: {SEARCH_37}')
print(f'LS 단어 빈도: {LS_WORD_FREQ}')
print(f'LS 형태소 빈도: {LS_MORPH_FREQ}')

37번 결과: /content/drive/MyDrive/DATA_2026/37_vowel_harmony_collision/search_results
LS 단어 빈도: /content/drive/MyDrive/DATA_2026/00_raw_data/02_nikl_ls/07_ALL_word_freq.csv
LS 형태소 빈도: /content/drive/MyDrive/DATA_2026/00_raw_data/02_nikl_ls/08_ALL_morpheme_freq.csv


## 2. 37번 결과 로드

In [3]:
# 37번 전체 결과 CSV 찾기 (가장 최신)
all_files = sorted(Path(SEARCH_37).glob('vowel_harmony_collision_all_*.csv'))
if not all_files:
    raise FileNotFoundError('37번 결과 CSV가 없습니다')

latest = all_files[-1]
print(f'사용 파일: {latest.name} ({latest.stat().st_size / 1024 / 1024:.1f}MB)')

df37 = pd.read_csv(latest, encoding='utf-8-sig')
print(f'37번 결과: {len(df37):,}행')
print(f'컬럼: {list(df37.columns)}')
print(f'\nenvironment 분포:')
print(df37['environment'].value_counts())
print(f'\ncollision_type 분포:')
print(df37['collision_type'].value_counts())

사용 파일: vowel_harmony_collision_all_20260312_071836.csv (22.9MB)
37번 결과: 93,413행
컬럼: ['word', 'pos', 'stem', 'conj_form', 'environment', 'stem_final_vowel_name', 'stem_ends_open', 'vowel_polarity', 'suffix_type', 'harmony_ok', 'collision_type', 'word_type', 'word_roman', 'pron_roman', 'conjugations', 'freq_LS_total', 'freq_MP_total', 'sense_no', 'definition']

environment 분포:
environment
vowel_collision    87271
vowel_harmony       6142
Name: count, dtype: int64

collision_type 분포:
collision_type
glide_insertion    84848
none                6142
deletion            1238
glide_formation     1185
Name: count, dtype: int64


In [4]:
# 어간(stem)과 기본형(word) 리스트 추출
stems = df37['stem'].dropna().unique().tolist()
words = df37['word'].dropna().unique().tolist()
print(f'고유 어간: {len(stems):,}개')
print(f'고유 기본형: {len(words):,}개')

# 활용형 리스트 추출 (conjugations 컬럼에서)
all_conj_forms = set()
for conj_str in df37['conjugations'].dropna():
    for form in str(conj_str).split(','):
        form = form.strip()
        if form:
            all_conj_forms.add(form)
print(f'고유 활용형: {len(all_conj_forms):,}개')

# 전체 검색 대상 (기본형 + 활용형)
all_search_forms = set(words) | all_conj_forms
print(f'전체 검색 대상: {len(all_search_forms):,}개')

고유 어간: 59,597개
고유 기본형: 59,597개
고유 활용형: 176,726개
전체 검색 대상: 236,322개


## 3. LS 단어 빈도 조회

In [5]:
# LS 단어 빈도 로드 (102MB)
print('LS 단어 빈도 로딩...')
df_ls_word = pd.read_csv(LS_WORD_FREQ, encoding='utf-8-sig', low_memory=False)
print(f'LS 단어 빈도: {len(df_ls_word):,}행')
print(f'컬럼: {list(df_ls_word.columns)}')
print(f'\ncorpus_type 분포:')
print(df_ls_word['corpus_type'].value_counts())

LS 단어 빈도 로딩...
LS 단어 빈도: 817,688행
컬럼: ['word_surface', 'morpheme_analysis', 'sense_analysis', 'corpus_type', 'corpus_name', 'freq', 'word_roman', 'word_roman_mfa']

corpus_type 분포:
corpus_type
NXLS    498684
SXLS    183641
MXLS    135363
Name: count, dtype: int64


In [6]:
# 기본형(word) 기준 빈도 조회 — word_surface에서 기본형 매칭
df_word_match = df_ls_word[df_ls_word['word_surface'].isin(all_search_forms)].copy()
print(f'기본형/활용형 매칭: {len(df_word_match):,}행')
print(f'매칭된 고유 어형: {df_word_match["word_surface"].nunique():,}개 / {len(all_search_forms):,}개')

# 코퍼스 유형별 빈도 피벗
freq_by_type = df_word_match.groupby(['word_surface', 'corpus_type'])['freq'].sum().reset_index()
freq_pivot = freq_by_type.pivot_table(
    index='word_surface', columns='corpus_type', values='freq', fill_value=0
).reset_index()
freq_pivot.columns.name = None

# 총 빈도
freq_cols = [c for c in freq_pivot.columns if c != 'word_surface']
freq_pivot['ls_word_freq_total'] = freq_pivot[freq_cols].sum(axis=1)

print(f'\n빈도 피벗: {len(freq_pivot):,}행')
print(freq_pivot.sort_values('ls_word_freq_total', ascending=False).head(20))

기본형/활용형 매칭: 12,228행
매칭된 고유 어형: 5,688개 / 236,322개

빈도 피벗: 5,688행
     word_surface   MXLS    NXLS    SXLS  ls_word_freq_total
4002           있는  201.0  6666.0  3998.0             10865.0
1625            때  212.0  2424.0  2712.0              5348.0
3726           위해   40.0  3385.0   100.0              3525.0
969            내가  412.0   341.0  2487.0              3240.0
4544          지난해    0.0  2775.0    24.0              2799.0
5432            해  115.0   521.0  2049.0              2685.0
1323           대해   14.0  2604.0    50.0              2668.0
5157           통해    1.0  2527.0    65.0              2593.0
3434           없는   68.0  1436.0   815.0              2319.0
3227          아니라   58.0  1205.0   831.0              2094.0
869             나  871.0   152.0  1038.0              2061.0
968             내  178.0  1001.0   742.0              1921.0
1352            데   18.0  1562.0   333.0              1913.0
1611           따라   27.0  1717.0   116.0              1860.0
4004           있어  18

## 4. LS 형태소 빈도 조회 (어기 빈도)

In [7]:
# LS 형태소 빈도 로드
print('LS 형태소 빈도 로딩...')
df_ls_morph = pd.read_csv(LS_MORPH_FREQ, encoding='utf-8-sig', low_memory=False)
print(f'LS 형태소 빈도: {len(df_ls_morph):,}행')
print(f'컬럼: {list(df_ls_morph.columns)}')
print(f'\npos 분포 (상위 10):')
print(df_ls_morph['pos'].value_counts().head(10))

LS 형태소 빈도 로딩...
LS 형태소 빈도: 146,028행
컬럼: ['word', 'sense_id', 'pos', 'corpus_type', 'corpus_name', 'freq', 'word_roman', 'word_roman_mfa']

pos 분포 (상위 10):
pos
NNG    85082
NNP    39953
VV     13141
VA      4649
NNB     1096
NR       954
NP       451
XR       369
VX       287
VCP       29
Name: count, dtype: int64


In [8]:
# 어간(stem) 기준 형태소 빈도 조회
# 용언 어간: VV(동사), VA(형용사) 품사의 word 컬럼에서 stem 매칭
verb_pos = df_ls_morph['pos'].isin(['VV', 'VA', 'VX'])
stem_set = set(stems)

# word 컬럼에서 어간 매칭 (어간 = 기본형에서 -다 제거)
df_stem_match = df_ls_morph[verb_pos & df_ls_morph['word'].isin(stem_set)].copy()
print(f'어간 매칭 (VV/VA/VX): {len(df_stem_match):,}행')
print(f'매칭된 고유 어간: {df_stem_match["word"].nunique():,}개 / {len(stem_set):,}개')

# 어간별 총 빈도
stem_freq = df_stem_match.groupby('word')['freq'].sum().reset_index()
stem_freq.columns = ['stem', 'ls_stem_freq']
print(f'\n어간 빈도 상위 20:')
print(stem_freq.sort_values('ls_stem_freq', ascending=False).head(20))

어간 매칭 (VV/VA/VX): 16,585행
매칭된 고유 어간: 3,789개 / 59,597개

어간 빈도 상위 20:
     stem  ls_stem_freq
3554    하         78066
2880    있         64784
1060    되         28025
1801    보         26446
0       가         17016
2434    않         15673
2586    없         14827
3064    주         14228
108     같         13389
362    그렇         12814
1466    먹         11454
2656    오         11152
3058    좋         10568
960    대하          9886
1700    받          7884
2834   이렇          7836
623    나오          7401
2789   위하          7218
1386    많          6968
1417    맞          6808


## 5. 37번 결과에 빈도 병합

In [9]:
# 1) 기본형(word) 기준 LS 단어 빈도 병합
word_freq_map = freq_pivot.set_index('word_surface')['ls_word_freq_total'].to_dict()
df37['ls_word_freq'] = df37['word'].map(word_freq_map).fillna(0).astype(int)

# 2) 어간(stem) 기준 LS 형태소 빈도 병합
stem_freq_map = stem_freq.set_index('stem')['ls_stem_freq'].to_dict()
df37['ls_stem_freq'] = df37['stem'].map(stem_freq_map).fillna(0).astype(int)

# 3) 활용형별 빈도 합산 (conjugations에 포함된 형태들)
def sum_conj_freq(conj_str):
    if pd.isna(conj_str):
        return 0
    total = 0
    for form in str(conj_str).split(','):
        form = form.strip()
        total += word_freq_map.get(form, 0)
    return total

df37['ls_conj_freq'] = df37['conjugations'].apply(sum_conj_freq)

print('빈도 병합 완료')
print(f'ls_word_freq > 0: {(df37["ls_word_freq"] > 0).sum():,}행 ({(df37["ls_word_freq"] > 0).mean()*100:.1f}%)')
print(f'ls_stem_freq > 0: {(df37["ls_stem_freq"] > 0).sum():,}행 ({(df37["ls_stem_freq"] > 0).mean()*100:.1f}%)')
print(f'ls_conj_freq > 0: {(df37["ls_conj_freq"] > 0).sum():,}행 ({(df37["ls_conj_freq"] > 0).mean()*100:.1f}%)')

빈도 병합 완료
ls_word_freq > 0: 3,454행 (3.7%)
ls_stem_freq > 0: 11,705행 (12.5%)
ls_conj_freq > 0: 12,513행 (13.4%)


In [10]:
# 빈도 상위 확인
print('=== 어간 빈도 상위 30 ===')
top30 = df37.nlargest(30, 'ls_stem_freq')[['word', 'stem', 'environment', 'collision_type',
                                             'stem_final_vowel_name', 'ls_word_freq',
                                             'ls_stem_freq', 'ls_conj_freq']]
print(top30.to_string(index=False))

=== 어간 빈도 상위 30 ===
word stem     environment  collision_type stem_final_vowel_name  ls_word_freq  ls_stem_freq  ls_conj_freq
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision glide_insertion                     ㅏ           234         78066        2775.0
  하다    하 vowel_collision 

## 6. 환경별 빈도 통계

In [11]:
# environment별 빈도 통계
print('=== environment별 어간 빈도 분포 ===')
for env in df37['environment'].unique():
    sub = df37[df37['environment'] == env]
    freq_col = sub['ls_stem_freq']
    print(f'\n{env} ({len(sub):,}행):')
    print(f'  빈도 > 0: {(freq_col > 0).sum():,}개 ({(freq_col > 0).mean()*100:.1f}%)')
    print(f'  빈도 > 10: {(freq_col > 10).sum():,}개')
    print(f'  빈도 > 100: {(freq_col > 100).sum():,}개')
    print(f'  평균: {freq_col.mean():.1f}, 중위수: {freq_col.median():.0f}')

=== environment별 어간 빈도 분포 ===

vowel_collision (87,271행):
  빈도 > 0: 8,863개 (10.2%)
  빈도 > 10: 5,478개
  빈도 > 100: 2,669개
  평균: 81.4, 중위수: 0

vowel_harmony (6,142행):
  빈도 > 0: 2,842개 (46.3%)
  빈도 > 10: 2,101개
  빈도 > 100: 1,254개
  평균: 552.2, 중위수: 0


In [12]:
# collision_type별 빈도 통계
print('=== collision_type별 어간 빈도 분포 ===')
for ct in df37['collision_type'].dropna().unique():
    sub = df37[df37['collision_type'] == ct]
    freq_col = sub['ls_stem_freq']
    print(f'\n{ct} ({len(sub):,}행):')
    print(f'  빈도 > 0: {(freq_col > 0).sum():,}개 ({(freq_col > 0).mean()*100:.1f}%)')
    print(f'  빈도 상위 5: {sub.nlargest(5, "ls_stem_freq")[["word", "ls_stem_freq"]].values.tolist()}')

=== collision_type별 어간 빈도 분포 ===

glide_insertion (84,848행):
  빈도 > 0: 7,350개 (8.7%)
  빈도 상위 5: [['하다', 78066], ['하다', 78066], ['하다', 78066], ['하다', 78066], ['하다', 78066]]

none (6,142행):
  빈도 > 0: 2,842개 (46.3%)
  빈도 상위 5: [['있다', 64784], ['있다', 64784], ['있다', 64784], ['있다', 64784], ['있다', 64784]]

glide_formation (1,185행):
  빈도 > 0: 733개 (61.9%)
  빈도 상위 5: [['오다', 11152], ['오다', 11152], ['오다', 11152], ['오다', 11152], ['오다', 11152]]

deletion (1,238행):
  빈도 > 0: 780개 (63.0%)
  빈도 상위 5: [['가다', 17016], ['가다', 17016], ['가다', 17016], ['가다', 17016], ['가다', 17016]]


In [13]:
# 어말모음별 빈도 통계
print('=== 어간말모음별 어간 빈도 분포 ===')
vowel_stats = df37.groupby('stem_final_vowel_name').agg(
    count=('ls_stem_freq', 'size'),
    has_freq=('ls_stem_freq', lambda x: (x > 0).sum()),
    mean_freq=('ls_stem_freq', 'mean'),
    max_freq=('ls_stem_freq', 'max'),
    total_freq=('ls_stem_freq', 'sum')
).sort_values('total_freq', ascending=False)
vowel_stats['pct_has_freq'] = (vowel_stats['has_freq'] / vowel_stats['count'] * 100).round(1)
print(vowel_stats.to_string())

=== 어간말모음별 어간 빈도 분포 ===
                       count  has_freq    mean_freq  max_freq  total_freq  pct_has_freq
stem_final_vowel_name                                                                  
ㅏ                      62991      3828    64.527345     78066     4064642           6.1
ㅣ                      11322      3841   187.237237     64784     2119900          33.9
ㅗ                       1185       665  1283.842194     26446     1521353          56.1
ㅚ                       7596       101    99.820300     28025      758235           1.3
ㅡ                       1651       949   426.484555      6160      704126          57.5
ㅓ                       2226       714   303.021114     14827      674525          32.1
ㅜ                       1285       772   274.826459     14228      353152          60.1
ㅐ                       4626       517    46.591224      4956      215531          11.2
ㅕ                        138        91   320.717391      2446       44259          65.9
ㅟ       

## 7. 결과 저장

In [14]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 빈도 보강 전체 결과 저장
out_path = f'{RESULT_DIR}/ls_freq_vowel_harmony_collision_{timestamp}.csv'
df37.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장: {out_path}')
print(f'  {len(df37):,}행, 컬럼: {list(df37.columns)}')

# 빈도 있는 것만 별도 저장 (분석용)
df37_freq = df37[df37['ls_stem_freq'] > 0].copy()
out_path2 = f'{RESULT_DIR}/ls_freq_vowel_collision_with_freq_{timestamp}.csv'
df37_freq.to_csv(out_path2, index=False, encoding='utf-8-sig')
print(f'\n빈도>0만: {out_path2}')
print(f'  {len(df37_freq):,}행')

print('\n저장 완료')

저장: /content/drive/MyDrive/DATA_2026/37_vowel_harmony_collision/search_results/ls_freq_vowel_harmony_collision_20260313_015158.csv
  93,413행, 컬럼: ['word', 'pos', 'stem', 'conj_form', 'environment', 'stem_final_vowel_name', 'stem_ends_open', 'vowel_polarity', 'suffix_type', 'harmony_ok', 'collision_type', 'word_type', 'word_roman', 'pron_roman', 'conjugations', 'freq_LS_total', 'freq_MP_total', 'sense_no', 'definition', 'ls_word_freq', 'ls_stem_freq', 'ls_conj_freq']

빈도>0만: /content/drive/MyDrive/DATA_2026/37_vowel_harmony_collision/search_results/ls_freq_vowel_collision_with_freq_20260313_015158.csv
  11,705행

저장 완료
